# Data-quality triage — counts, severities, owning workflow

> Explore and compute freely: nothing computed outside Headway's calculation library (services/calc) can ever become a reported figure. Only the calculation library writes computed.metric_values, and the walls are structural database CHECKs, not policy.

Headway **fails loudly**: gaps, conflicts between sources, and validation
failures are never silently dropped or interpolated — each becomes a
`dq.issues` row with a type, a severity, an owner, and a resolution
workflow, because an unexplained gap becomes a finding in an FTA triennial
review. This notebook is the analyst's view of that queue.

**You need:** `HEADWAY_API_URL`, plus `HEADWAY_USERNAME` /
`HEADWAY_PASSWORD` for a Headway account. The data-quality endpoints
currently accept only a signed-in session (not a machine key) — the client
says so honestly, and `headway_client.login` wraps the API's own login.
Credentials come from the environment; never write them into a notebook.

*This notebook was executed once against a live Headway demo stack (2026-07-15) so its outputs are real; some demo figures come from simulated feeds and say so in their provenance. To run it yourself you need a running Headway API and the environment variables named below — credentials are never written into a notebook.*

In [1]:
import os

import pandas as pd

from headway_client import HeadwayClient, frames, login

pd.set_option("display.width", 160)

base_url = os.environ.get("HEADWAY_API_URL", "http://127.0.0.1:8000")
session_token = login(
    base_url, os.environ["HEADWAY_USERNAME"], os.environ["HEADWAY_PASSWORD"]
)
hw = HeadwayClient(base_url, token=session_token)

## The headline counts

`dq_issue_counts` is computed server-side over exactly the same rows the
issue list serves, so a summary card can never disagree with the table
under it. Missing severities/statuses appear as explicit zeros.

In [2]:
counts = hw.dq_issue_counts()
print(f"total issues: {counts.total}")
print(f"by severity:  {counts.by_severity}")
print(f"by status:    {counts.by_status}")

total issues: 35456
by severity:  {'blocking': 279, 'warning': 31869, 'info': 3308}
by status:    {'open': 35210, 'owned': 0, 'resolved': 246}


## Severity × status — where the work is

Severity vocabulary: `blocking` (stands between the agency and a
certifiable figure), `warning`, `info`. Status: `open`, `owned`,
`resolved`.

In [3]:
issues = frames.dq_issues_frame(hw.dq_issues())
pd.crosstab(issues["severity"], issues["status"], margins=True)

status,open,resolved,All
severity,,,
blocking,33,246,279
info,3308,0,3308
warning,31869,0,31869
All,35210,246,35456


## What kinds of problems, and whose they are

Every issue has a type (the writers are the transform's quarantine path,
the calc library's exclusions/refusals, and the AI anomaly flagger — which
writes *only* flags, never figures) and the owning-workflow fields:
`owner`, `status`, and on resolution the plain-language `resolution` text
plus optional `resolution_minutes` (kept as a nullable integer — an
unmeasured effort is missing, never zero).

In [4]:
issues.groupby("issue_type", dropna=False).agg(
    count=("issue_id", "size"),
    blocking=("severity", lambda s: int((s == "blocking").sum())),
    open=("status", lambda s: int((s == "open").sum())),
).sort_values("count", ascending=False).head(12)

,count,blocking,open
issue_type,,,
telemetry_gap_excluded,21800,0,21800
pmt_invalid_trip_excluded,6494,0,6494
block_unavailable,3235,0,3235
layover_exceeds_max,1565,0,1565
apc_count_imbalance,1293,0,1293
apc_negative_load,660,0,660
telemetry_gap,244,244,0
simulated_source_data,62,0,62
coverage_below_threshold,20,20,18


In [5]:
print("ownership:")
print(issues["owner"].fillna("(unassigned)").value_counts().to_string())
print()
resolved = issues[issues["status"] == "resolved"]
print(f"resolved: {len(resolved)}; with measured effort: "
      f"{int(resolved['resolution_minutes'].notna().sum())}")

ownership:
owner
(unassigned)    35456

resolved: 246; with measured effort: 0


## The triage view: open blocking issues first

A `blocking` issue is exactly what it sounds like: the certification
cockpit will not arm while one is open in the NTD category. Titles and
descriptions are written in plain language for a transit operations
manager, and `source_record_ids` tie an issue to the exact raw records
behind it — the same fail-loudly provenance as everywhere else.

In [6]:
open_blocking = issues[
    (issues["status"] == "open") & (issues["severity"] == "blocking")
]
print(f"{len(open_blocking)} open blocking issue(s)")
open_blocking[["issue_type", "title", "owner", "created_at"]].head(8)

33 open blocking issue(s)


,issue_type,title,owner,created_at
3157,apc_missing_trips_above_fta_threshold,Missing-trip share 0.1869 exceeds the FTA 2% t...,None,2026-07-10 16:08:38.820380+00:00
5684,apc_missing_trips_above_fta_threshold,Missing-trip share 0.2991 exceeds the FTA 2% t...,None,2026-07-10 16:08:46.729471+00:00
8823,coverage_below_threshold,Coverage 0.9146 below threshold 0.95: 799 of 9...,None,2026-07-11 07:09:58.243935+00:00
9180,coverage_below_threshold,Coverage 0.9146 below threshold 0.95: 799 of 9...,None,2026-07-11 07:09:58.243935+00:00
12543,apc_missing_trips_above_fta_threshold,Missing-trip share 0.3991 exceeds the FTA 2% t...,None,2026-07-11 21:28:25.834657+00:00
16926,coverage_below_threshold,Coverage 0.7025 below threshold 0.90: 277 of 9...,None,2026-07-11 21:28:25.834657+00:00
18093,coverage_below_threshold,Coverage 0.7025 below threshold 0.90: 277 of 9...,None,2026-07-11 21:28:25.834657+00:00
18343,coverage_below_threshold,Coverage 0.8580 below threshold 0.90: 377 of 2...,None,2026-07-11 21:28:25.834657+00:00


Resolving an issue is a state change with an audit trail, done in
the Headway app (or its API) by a data steward — deliberately not from a
notebook. What this notebook is for: seeing the queue whole, spotting
patterns by type/route/source, and deciding where the next hour of data
stewardship goes.

**See also:** [01-ridership-exploration](01-ridership-exploration.ipynb) ·
[02-otp-headway-adherence](02-otp-headway-adherence.ipynb) ·
[docs/analyst-access.md](../docs/analyst-access.md).